# 07 - Agentic Retrieval with Knowledge Bases

Goal: Implement **Azure AI Search Knowledge Bases** for agentic retrieval using real enterprise data.

**Scenario: HR Benefits Assistant**
- **Knowledge Source 1**: Health benefits documentation (Northwind Health Plus plans)
- **Knowledge Source 2**: HR policies and company information (Zava company docs)
- **Use Case**: Employees ask questions about benefits, perks, policies, and company information

**What this notebook demonstrates:**
1. Creates **Knowledge Sources** that reference search indices with real data
2. Creates **Knowledge Bases** that orchestrate agentic retrieval
3. Configures **Azure OpenAI integration** for answer synthesis
4. Implements **retrieval and answer instructions** for LLM query planning
5. Demonstrates **automatic source selection** and **answer generation**

**Prerequisites:**
- **Run [05-azure-infra-setup.ipynb](./05-azure-infra-setup.ipynb) first** to create search service and AI Foundry
- `.env` configured with: `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZURE_SEARCH_SERVICE_NAME`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_DEPLOYMENT`

**API Version:** 2025-11-01-preview

**Data Source:** Sample data from `data/index-data/` directory (384 documents total)

**References:**
- [How to create a knowledge base](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Azure AI Search agentic retrieval](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-overview)

## ⚠️ Important Setup Notes

**This notebook follows the Microsoft LAB511 pattern for knowledge base creation.**

### Prerequisites

1. **Azure OpenAI API Key**: Ensure `AZURE_OPENAI_API_KEY` is set in your `.env` file
2. **Vectorizer Configuration**: The search indices use Azure OpenAI for embeddings
3. **RBAC Permissions**: Agent blueprint needs `Cognitive Services OpenAI User` role

### Common Issues & Fixes

**401 Unauthorized (Vectorization)**:
```
Error: "Could not complete vectorization action. The vectorization endpoint returned status code '401'"
```
**Fix**: Run `python update-index-vectorizers.py` to update index vectorizers with current API key

**400 Bad Request (Invalid kind)**:
```
Error: "The specified kind '' is not valid"
```
**Fix**: Ensure all `knowledgeSourceParams` include `"kind": "searchIndex"`

**No Results (Agent reasoning not querying sources)**:
```
Response: "Sorry, I could not find an answer for your query"
```
**Fix**: Add `"alwaysQuerySource": True` to force querying all knowledge sources

### Key Differences from REST API Pattern

This notebook uses REST API calls instead of the Python SDK. Key considerations:

1. **No `searchFields`**: Don't specify searchFields - let it search all searchable fields by default
2. **Required `kind` parameter**: All knowledge source params need `"kind": "searchIndex"`
3. **`alwaysQuerySource`**: Recommended for predictable behavior
4. **Simplified first**: Start with minimal knowledge base config, add complexity later

### References

- [Microsoft LAB511 GitHub](https://github.com/microsoft/ignite25-LAB511-build-agentic-knowledge-bases-next-level-rag-with-azure-ai-search)
- [FINAL-SOLUTION-SUMMARY.md](FINAL-SOLUTION-SUMMARY.md) - Detailed fix documentation
- [COMPARISON-MSFT-LAB511.md](COMPARISON-MSFT-LAB511.md) - Pattern comparison

In [11]:
import os
import subprocess
import requests
import json
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient

load_dotenv(override=True)

# Load configuration
subscription_id = os.getenv('AZURE_SUBSCRIPTION_ID')
resource_group = os.getenv('AZURE_RESOURCE_GROUP', 'rg-agent-identity-sandbox')
search_service_name = os.getenv('AZURE_SEARCH_SERVICE_NAME', '')
azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')  # Remove trailing slash
azure_openai_api_key = os.getenv('AZURE_OPENAI_API_KEY', '')
azure_openai_deployment = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
azure_openai_model = os.getenv('AZURE_OPENAI_MODEL', 'gpt-4o')

# Fix PATH for Azure CLI
az_paths = ['/usr/local/bin', '/opt/homebrew/bin', '/usr/bin']
for path in az_paths:
    if path not in os.environ['PATH']:
        os.environ['PATH'] = f"{path}:{os.environ['PATH']}"

# Validate required configuration
required = {
    'AZURE_SEARCH_SERVICE_NAME': search_service_name,
    'AZURE_OPENAI_ENDPOINT': azure_openai_endpoint,
    'AZURE_OPENAI_API_KEY': azure_openai_api_key,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"❌ Missing in .env: {', '.join(missing)}")

# Construct endpoints
search_endpoint = f"https://{search_service_name}.search.windows.net"

# Get search admin key
result = subprocess.run(
    f"az search admin-key show --resource-group {resource_group} --service-name {search_service_name} --query primaryKey --output tsv",
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    search_api_key = result.stdout.strip()
else:
    raise RuntimeError(f"Could not retrieve search admin key: {result.stderr}")

# Initialize clients
index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=AzureKeyCredential(search_api_key)
)

# API version for knowledge bases (preview)
api_version = "2025-11-01-preview"

print('✅ Configuration loaded')
print(f'   Search Service: {search_service_name}')
print(f'   Search Endpoint: {search_endpoint}')
print(f'   Azure OpenAI Endpoint: {azure_openai_endpoint}')
print(f'   Azure OpenAI Deployment: {azure_openai_deployment}')
print(f'   API Version: {api_version}')

✅ Configuration loaded
   Search Service: a365-search-6uuruydd4tej6
   Search Endpoint: https://a365-search-6uuruydd4tej6.search.windows.net
   Azure OpenAI Endpoint: https://aifz6vuv4jvgmejw.openai.azure.com
   Azure OpenAI Deployment: gpt-4o
   API Version: 2025-11-01-preview


## Step 1: Create Indices and Load Sample Data

Create search indices for health benefits and HR documents, then load data from the `data/index-data/` directory.

In [12]:
import json
from azure.search.documents import SearchClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch
)

print("📋 Creating search indices for knowledge base...\n")

# Define index schema matching the data structure
def create_index_schema(index_name: str):
    """Create index schema with vector search and semantic search."""
    
    fields = [
        SimpleField(name="uid", type="Edm.String", key=True, filterable=True, sortable=True),
        SimpleField(name="snippet_parent_id", type="Edm.String", filterable=True),
        SimpleField(name="blob_path", type="Edm.String", filterable=True, retrievable=True),
        SearchableField(name="snippet", type="Edm.String"),
        SearchField(
            name="snippet_vector",
            type="Collection(Edm.Single)",
            searchable=True,
            vector_search_dimensions=3072,
            vector_search_profile_name="vector-profile"
        )
    ]
    
    # Vector search configuration
    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="hnsw-algorithm",
                parameters={
                    "m": 4,
                    "efConstruction": 400,
                    "efSearch": 500,
                    "metric": "cosine"
                }
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="vector-profile",
                algorithm_configuration_name="hnsw-algorithm",
                vectorizer_name="openai-vectorizer"
            )
        ],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="openai-vectorizer",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=azure_openai_endpoint,
                    deployment_name="text-embedding-3-large",
                    model_name="text-embedding-3-large",
                    api_key=azure_openai_api_key
                )
            )
        ]
    )
    
    # Semantic search configuration
    semantic_config = SemanticConfiguration(
        name="semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            content_fields=[SemanticField(field_name="snippet")]
        )
    )
    
    semantic_search = SemanticSearch(
        configurations=[semantic_config],
        default_configuration_name="semantic-config"
    )
    
    return SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search,
        semantic_search=semantic_search
    )

# Create indices
indices_to_create = ["healthdocs-index", "hrdocs-index"]

for index_name in indices_to_create:
    try:
        index = create_index_schema(index_name)
        index_client.create_or_update_index(index)
        print(f"✅ Created index: {index_name}")
        print(f"   Vector search: Enabled (3072 dims)")
        print(f"   Semantic search: Enabled")
    except Exception as e:
        print(f"⚠️  Error with {index_name}: {str(e)[:150]}")
    print()

# Load data from JSONL files
print("📤 Loading sample data from ../data/index-data/...\n")

# Use relative path from notebooks directory
data_files = [
    ("healthdocs-index", "../data/index-data/healthdocs-exported.jsonl"),
    ("hrdocs-index", "../data/index-data/hrdocs-exported.jsonl")
]

for index_name, file_path in data_files:
    try:
        # Read JSONL file
        documents = []
        with open(file_path, 'r') as f:
            for line in f:
                doc = json.loads(line.strip())
                documents.append(doc)
        
        print(f"📄 Loaded {len(documents)} documents from {file_path.split('/')[-1]}")
        
        # Upload in batches of 100
        client = SearchClient(
            endpoint=search_endpoint,
            index_name=index_name,
            credential=AzureKeyCredential(search_api_key)
        )
        
        batch_size = 100
        total_uploaded = 0
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i+batch_size]
            result = client.upload_documents(documents=batch)
            succeeded = sum(1 for r in result if r.succeeded)
            total_uploaded += succeeded
            print(f"   Batch {i//batch_size + 1}: {succeeded}/{len(batch)} documents uploaded")
        
        print(f"✅ Total: {total_uploaded}/{len(documents)} documents uploaded to {index_name}")
        print()
        
    except FileNotFoundError:
        print(f"⚠️  File not found: {file_path}")
        print(f"   Make sure you're running this notebook from the notebooks/ directory")
        print(f"   Skipping {index_name}")
        print()
    except Exception as e:
        print(f"⚠️  Error loading {index_name}: {str(e)[:150]}")
        print()

print("✅ Indices created and populated with sample data")

📋 Creating search indices for knowledge base...

✅ Created index: healthdocs-index
   Vector search: Enabled (3072 dims)
   Semantic search: Enabled

✅ Created index: hrdocs-index
   Vector search: Enabled (3072 dims)
   Semantic search: Enabled

📤 Loading sample data from ../data/index-data/...

📄 Loaded 334 documents from healthdocs-exported.jsonl
   Batch 1: 100/100 documents uploaded
   Batch 2: 100/100 documents uploaded
   Batch 3: 100/100 documents uploaded
   Batch 4: 34/34 documents uploaded
✅ Total: 334/334 documents uploaded to healthdocs-index

📄 Loaded 50 documents from hrdocs-exported.jsonl
   Batch 1: 50/50 documents uploaded
✅ Total: 50/50 documents uploaded to hrdocs-index

✅ Indices created and populated with sample data


## Step 2: Create Knowledge Sources

Knowledge sources are references to search indices that the knowledge base will query.

In [13]:
print("📚 Creating knowledge sources...\n")

# Knowledge sources reference the search indices
knowledge_sources = [
    {
        "name": "health-benefits-ks",
        "description": "Health insurance benefits documentation including Northwind Health Plus plans, coverage details, copayments, and enrollment information",
        "index_name": "healthdocs-index"
    },
    {
        "name": "hr-policies-ks",
        "description": "HR policies and company information including Zava company overview, employee handbook, perks, vacation policies, and role library",
        "index_name": "hrdocs-index"
    }
]

# Create knowledge sources using REST API
for ks in knowledge_sources:
    knowledge_source_definition = {
        "kind": "searchIndex",
        "name": ks["name"],
        "description": ks["description"],
        "searchIndexParameters": {  # No searchFields - searches all searchable fields by default
            "searchIndexName": ks["index_name"],
            "sourceDataFields": [
                {"name": "snippet"},
                {"name": "uid"},
                {"name": "blob_path"}
            ],
            "semanticConfigurationName": "semantic-config"  # Enable semantic search
        }
    }
    
    url = f"{search_endpoint}/knowledgesources/{ks['name']}?api-version={api_version}"
    headers = {
        "Content-Type": "application/json",
        "api-key": search_api_key
    }
    
    response = requests.put(url, json=knowledge_source_definition, headers=headers)
    
    if response.status_code in [200, 201, 204]:
        print(f"✅ Created knowledge source: {ks['name']}")
        print(f"   Description: {ks['description'][:80]}...")
        print(f"   Index: {ks['index_name']}")
        print(f"   Semantic search: Enabled")
    else:
        print(f"⚠️  Error creating {ks['name']}: {response.status_code}")
        print(f"   {response.text}")
    print()

print("✅ Knowledge sources created")

📚 Creating knowledge sources...

✅ Created knowledge source: health-benefits-ks
   Description: Health insurance benefits documentation including Northwind Health Plus plans, c...
   Index: healthdocs-index
   Semantic search: Enabled

✅ Created knowledge source: hr-policies-ks
   Description: HR policies and company information including Zava company overview, employee ha...
   Index: hrdocs-index
   Semantic search: Enabled

✅ Knowledge sources created


## Step 3: Create Knowledge Base

Create a knowledge base that orchestrates retrieval across knowledge sources with Azure OpenAI integration.

In [14]:
print("🧠 Creating knowledge base with Azure OpenAI integration...\n")

# Define knowledge base for HR benefits assistant
knowledge_base_definition = {
    "name": "hr-benefits-assistant",
    "description": "HR Benefits Assistant knowledge base for answering employee questions about health benefits, company policies, perks, and HR procedures",
    "retrievalInstructions": """You are an HR Benefits Assistant for employees at Zava company.

Use the health-benefits-ks knowledge source for questions about:
- Health insurance plans (Northwind Health Plus, Northwind Standard)
- Medical coverage, copayments, deductibles
- Enrollment periods and eligibility
- Provider networks and claims

Use the hr-policies-ks knowledge source for questions about:
- Company policies and procedures
- Vacation time and perks
- Employee benefits and programs
- Role descriptions and responsibilities
- Company history and values

For questions spanning both areas, query both sources and synthesize a comprehensive answer.""",
    
    "answerInstructions": """Provide clear, employee-friendly answers based on the retrieved documents. 

Guidelines:
- Be concise but complete (2-3 sentences for simple questions, more for complex ones)
- Include specific details like dollar amounts, timeframes, and requirements
- If information is not in the knowledge base, clearly state that
- Use a helpful, professional tone
- Cite specific plan names or policy sections when relevant""",
    
    "outputMode": "answerSynthesis",
    
    "knowledgeSources": [
        {"name": "health-benefits-ks"},
        {"name": "hr-policies-ks"}
    ],
    
    "models": [
        {
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
                "resourceUri": azure_openai_endpoint,
                "apiKey": azure_openai_api_key,
                "deploymentId": azure_openai_deployment,
                "modelName": azure_openai_model
            }
        }
    ],
    
    "retrievalReasoningEffort": {
        "kind": "low"
    }
}

# Create knowledge base using REST API
url = f"{search_endpoint}/knowledgebases/hr-benefits-assistant?api-version={api_version}"
headers = {
    "Content-Type": "application/json",
    "api-key": search_api_key
}

response = requests.put(url, json=knowledge_base_definition, headers=headers)

if response.status_code in [200, 201, 204]:
    print("✅ Knowledge base created successfully!")
    print(f"   Name: hr-benefits-assistant")
    print(f"   Scenario: HR Benefits Assistant for Zava employees")
    print(f"   Output mode: answerSynthesis")
    print(f"   Knowledge sources: 2 (health benefits + HR policies)")
    print(f"   LLM: {azure_openai_deployment} ({azure_openai_model})")
    print(f"   Reasoning effort: low")
else:
    print(f"⚠️  Error creating knowledge base: {response.status_code}")
    print(f"   {response.text}")

print("\n✅ HR Benefits Assistant ready for employee queries")

🧠 Creating knowledge base with Azure OpenAI integration...

✅ Knowledge base created successfully!
   Name: hr-benefits-assistant
   Scenario: HR Benefits Assistant for Zava employees
   Output mode: answerSynthesis
   Knowledge sources: 2 (health benefits + HR policies)
   LLM: gpt-4o (gpt-4o)
   Reasoning effort: low

✅ HR Benefits Assistant ready for employee queries


## Step 4: Query the Knowledge Base

Query the knowledge base using the retrieval API. The LLM will automatically select the appropriate knowledge source and synthesize an answer.

In [15]:
def query_knowledge_base(query: str, kb_name: str = "hr-benefits-assistant"):
    """Query the HR Benefits Assistant knowledge base."""
    
    # Build retrieval request
    retrieval_request = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": query
                    }
                ]
            }
        ],
        "knowledgeSourceParams": [
            {
                "kind": "searchIndex",
                "knowledgeSourceName": "health-benefits-ks",
                "includeReferences": True,
                "includeReferenceSourceData": True,
                "alwaysQuerySource": True
            },
            {
                "kind": "searchIndex",
                "knowledgeSourceName": "hr-policies-ks",
                "includeReferences": True,
                "includeReferenceSourceData": True,
                "alwaysQuerySource": True
            }
        ],
        "includeActivity": True
    }
    
    # Query knowledge base
    url = f"{search_endpoint}/knowledgebases/{kb_name}/retrieve?api-version={api_version}"
    headers = {
        "Content-Type": "application/json",
        "api-key": search_api_key
    }
    
    response = requests.post(url, json=retrieval_request, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        return {
            "error": f"Status {response.status_code}: {response.text}"
        }

def display_kb_response(query: str, response: dict):
    """Display knowledge base response in a readable, color-coded format."""
    
    print("\n" + "="*80)
    print(f"❓ QUERY: {query}")
    print("="*80)
    
    # Check for HTTP-level errors
    if "error" in response and isinstance(response["error"], str):
        print("\n❌ HTTP ERROR")
        print("-"*80)
        print(response["error"])
        print("="*80)
        return
    
    # Extract activity to check for errors
    activity = response.get("activity", [])
    has_errors = False
    error_details = []
    
    for act in activity:
        if "error" in act:
            has_errors = True
            error_details.append({
                "source": act.get("knowledgeSourceName", "unknown"),
                "type": act.get("type", "unknown"),
                "message": act["error"].get("message", "Unknown error")[:200]
            })
    
    # Display status banner
    if has_errors:
        print("\n⚠️  STATUS: PARTIAL FAILURE - Some queries failed")
        print("-"*80)
        
        # Show errors clearly
        print("\n🔴 ERRORS ENCOUNTERED:")
        for i, err in enumerate(error_details, 1):
            print(f"\n   Error {i}: {err['source']} ({err['type']})")
            
            # Parse specific error types
            if "vectorization" in err['message'].lower() and "401" in err['message']:
                print("   Issue: ❌ Vectorization failed - API key unauthorized")
                print("   Fix: Run 'python update-index-vectorizers.py'")
            elif "400" in err['message'] and "invalid kind" in err['message'].lower():
                print("   Issue: ❌ Invalid knowledge source parameter")
                print("   Fix: Ensure 'kind': 'searchIndex' is specified")
            else:
                print(f"   Message: {err['message']}")
    else:
        print("\n✅ STATUS: SUCCESS - All queries completed")
        print("-"*80)
    
    # Extract and display answer
    answer_found = False
    if "response" in response and response["response"]:
        answer_content = response["response"][0].get("content", [])
        if answer_content:
            answer_text = answer_content[0].get("text", "")
            if answer_text and "sorry" not in answer_text.lower():
                answer_found = True
                print("\n💬 ANSWER:")
                print("-"*80)
                print(answer_text)
            elif answer_text:
                print("\n⚠️  NO ANSWER FOUND:")
                print("-"*80)
                print(answer_text)
    
    # Extract and display references (only if we have an answer)
    if answer_found and "references" in response and response["references"]:
        print(f"\n📚 SOURCES ({len(response['references'])} citations):")
        print("-"*80)
        for i, ref in enumerate(response["references"][:5], 1):  # Show first 5
            ref_id = ref.get("referenceId", f"ref-{i}")
            snippet = ref.get("snippet", "")[:150]
            blob_path = ref.get("blob_path", "Unknown")
            filename = blob_path.split('/')[-1] if blob_path else "Unknown"
            
            print(f"\n   [{ref_id}] {filename}")
            print(f"   {snippet}...")
        
        if len(response["references"]) > 5:
            print(f"\n   ... and {len(response['references']) - 5} more sources")
    
    # Display query activity
    if activity:
        print(f"\n🔍 QUERY ACTIVITY:")
        print("-"*80)
        
        # Group by type
        planning = [a for a in activity if a.get("type") == "modelQueryPlanning"]
        searches = [a for a in activity if a.get("type") == "searchIndex"]
        reasoning = [a for a in activity if a.get("type") == "agenticReasoning"]
        synthesis = [a for a in activity if a.get("type") == "modelAnswerSynthesis"]
        
        if planning:
            p = planning[0]
            print(f"   1. Query Planning: {p.get('inputTokens', 0)} tokens → {p.get('outputTokens', 0)} tokens ({p.get('elapsedMs', 0)}ms)")
        
        if searches:
            print(f"   2. Knowledge Source Queries: {len(searches)} searches")
            for s in searches:
                status = "❌" if "error" in s else "✅"
                ks_name = s.get("knowledgeSourceName", "unknown")
                count = s.get("count", 0)
                search_arg = s.get("searchIndexArguments", {}).get("search", "")[:50]
                print(f"      {status} {ks_name}: {count} results ('{search_arg}...')")
        
        if reasoning:
            r = reasoning[0]
            effort = r.get("retrievalReasoningEffort", {}).get("kind", "unknown")
            print(f"   3. Agentic Reasoning: effort={effort}")
        
        if synthesis:
            s = synthesis[0]
            print(f"   4. Answer Synthesis: {s.get('inputTokens', 0)} tokens → {s.get('outputTokens', 0)} tokens ({s.get('elapsedMs', 0)}ms)")
    
    # Final summary
    print("\n" + "="*80)
    if has_errors:
        print("❌ RESULT: Query failed - see errors above")
    elif answer_found:
        print("✅ RESULT: Answer successfully generated with citations")
    else:
        print("⚠️  RESULT: Query succeeded but no relevant information found")
    print("="*80 + "\n")

print("✅ Knowledge base query functions ready")


✅ Knowledge base query functions ready


## Step 5: Demo Queries

Test the knowledge base with various queries to see automatic source selection and answer synthesis.

In [16]:
print("🧪 Testing HR Benefits Assistant with Employee Queries\n")

# Test 1: Health benefits question
print("\n1️⃣ Test: Health Benefits Question")
query1 = "What are the copayment amounts for office visits with Northwind Health Plus?"
response1 = query_knowledge_base(query1)
display_kb_response(query1, response1)

import time
time.sleep(2)  # Avoid rate limiting

🧪 Testing HR Benefits Assistant with Employee Queries


1️⃣ Test: Health Benefits Question

❓ QUERY: What are the copayment amounts for office visits with Northwind Health Plus?

✅ STATUS: SUCCESS - All queries completed
--------------------------------------------------------------------------------

💬 ANSWER:
--------------------------------------------------------------------------------
For office visits under the Northwind Health Plus plan, copayment amounts vary depending on the type of provider. Office visits with primary care physicians have a copayment of $35, while visits with specialists require a $60 copayment. Mental health visits with a psychiatrist or another mental health provider have a copayment of $45 [ref_id:1][ref_id:3].

📚 SOURCES (10 citations):
--------------------------------------------------------------------------------

   [ref-1] Unknown
   ...

   [ref-2] Unknown
   ...

   [ref-3] Unknown
   ...

   [ref-4] Unknown
   ...

   [ref-5] Unknown
   ...

   .

In [17]:
# Test 2: Company/HR policy question
print("\n2️⃣ Test: Company Policy Question")
query2 = "How many weeks of vacation do employees get at Zava?"
response2 = query_knowledge_base(query2)
display_kb_response(query2, response2)

time.sleep(2)  # Avoid rate limiting


2️⃣ Test: Company Policy Question

❓ QUERY: How many weeks of vacation do employees get at Zava?

✅ STATUS: SUCCESS - All queries completed
--------------------------------------------------------------------------------

💬 ANSWER:
--------------------------------------------------------------------------------
Employees at Zava receive different vacation benefits based on their tier. Standard tier employees get 2 weeks of vacation along with a health and wellness stipend. Senior tier employees receive 4 weeks of vacation and travel vouchers for a dream destination. Executive tier employees enjoy 6 weeks of vacation, which includes a luxury resort getaway with family [ref_id:0][ref_id:1].

📚 SOURCES (11 citations):
--------------------------------------------------------------------------------

   [ref-1] Unknown
   ...

   [ref-2] Unknown
   ...

   [ref-3] Unknown
   ...

   [ref-4] Unknown
   ...

   [ref-5] Unknown
   ...

   ... and 6 more sources

🔍 QUERY ACTIVITY:
------------

In [18]:
# Test 3: Cross-domain question (both health benefits and company info)
print("\n3️⃣ Test: Cross-Domain Question")
query3 = "Tell me about Zava's health benefits and employee perks"
response3 = query_knowledge_base(query3)
display_kb_response(query3, response3)

print("\n✅ HR Benefits Assistant query tests complete!")
print("\n📊 Summary:")
print("   • Successfully queried knowledge base across multiple sources")
print("   • LLM automatically selected appropriate knowledge sources")
print("   • Answers synthesized from 384 document snippets")
print("   • Vector search + semantic ranking for high relevance")


3️⃣ Test: Cross-Domain Question

❓ QUERY: Tell me about Zava's health benefits and employee perks

✅ STATUS: SUCCESS - All queries completed
--------------------------------------------------------------------------------

💬 ANSWER:
--------------------------------------------------------------------------------
Zava offers its employees two health insurance plans through Northwind Health: Northwind Health Plus and Northwind Standard. Northwind Health Plus provides comprehensive coverage, including medical, vision, dental, prescription drugs, mental health, substance abuse, and emergency services, both in-network and out-of-network [ref_id:0][ref_id:3]. Northwind Standard offers basic coverage for medical, vision, and dental services, as well as preventive care and prescription drugs, but does not cover emergency services or out-of-network care [ref_id:0][ref_id:1]. Employees are responsible for paying premiums, which are deducted from their paychecks [ref_id:6]. 

In addition to heal

In [19]:
# Try a simple query to test if content filter is the issue
print("\n4️⃣ Test: Simple Query")
query4 = "What is Zava company?"
response4 = query_knowledge_base(query4)
display_kb_response(query4, response4)


4️⃣ Test: Simple Query

❓ QUERY: What is Zava company?

✅ STATUS: SUCCESS - All queries completed
--------------------------------------------------------------------------------

💬 ANSWER:
--------------------------------------------------------------------------------
Zava is a company founded in 1985, known for its pioneering role in the consumer electronics industry. It has achieved several milestones, including launching the first handheld personal computer in 1990 and expanding into sustainable product lines in 2015 [ref_id:0]. Zava is also a leader in the aerospace industry, providing advanced electronic components for commercial and military aircraft [ref_id:2]. The company values innovation, diversity, and sustainability, and strives to create a dynamic and inclusive workplace [ref_id:0][ref_id:4]. 

Zava offers various employee benefits, including vacation perks with tiers ranging from 2 to 6 weeks, depending on the employee's level, and health and wellness stipends [ref_id:

## Summary: Knowledge Base Architecture

### Scenario: HR Benefits Assistant for Zava Employees

Employees can ask questions about:
- Health insurance (Northwind Health Plus, Standard)
- Company policies and procedures
- Vacation time and perks
- Role descriptions and responsibilities

---

### Data Architecture

**1️⃣ Search Indices (Data Layer)**
| Index | Documents | Content |
|-------|-----------|---------|
| `healthdocs-index` | 334 | Northwind health benefits |
| `hrdocs-index` | 50 | Zava company info |

- **Total**: 384 document snippets
- **Vector search**: text-embedding-3-large (3072 dims)
- **Semantic search**: Enabled

**2️⃣ Knowledge Sources (Abstraction Layer)**
- `health-benefits-ks` → healthdocs-index
- `hr-policies-ks` → hrdocs-index

**3️⃣ Knowledge Base (Orchestration Layer)**
- `hr-benefits-assistant`
- Integrates 2 knowledge sources
- Connected to Azure OpenAI for answer synthesis

---

### Query Flow

```
Employee Question
    ↓
Knowledge Base (hr-benefits-assistant)
    ↓
LLM analyzes query + retrieval instructions
    ↓
Selects appropriate Knowledge Source(s)
    ├─ health-benefits-ks (for benefits questions)
    └─ hr-policies-ks (for policy questions)
    ↓
Vector + Semantic Search on Index(es)
    ↓
Retrieves relevant document snippets
    ↓
LLM synthesizes employee-friendly answer
    ↓
Returns answer + sources + activity log
```

---

### Key Features Demonstrated

- ✅ Real enterprise data (384 documents)
- ✅ Vector search with embeddings (3072 dimensions)
- ✅ Semantic ranking for improved relevance
- ✅ Automatic source selection based on query
- ✅ LLM-powered answer synthesis
- ✅ Citation tracking with source attribution
- ✅ Activity logging (transparency)
- ✅ Multi-domain knowledge integration

---

### Next Steps for Production

1. Add document-level security (permission filters from notebook 06)
2. Integrate with employee authentication (Azure AD)
3. Add more knowledge sources (policies, procedures, FAQs)
4. Fine-tune retrieval/answer instructions for your domain
5. Implement conversation history for multi-turn dialogs
6. Add feedback collection to improve responses
7. Monitor query patterns and optimize indexing strategy

---

### References

- [Azure AI Search Agentic Retrieval](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-overview)
- [How to Create a Knowledge Base](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Microsoft LAB511 - Agentic Knowledge Bases](https://github.com/microsoft/ignite25-LAB511-build-agentic-knowledge-bases-next-level-rag-with-azure-ai-search) (reference content)